# 10. GGUF Quantization & Deployment

LoRA adapter를 base model에 병합하고 GGUF 형식으로 변환합니다.
다양한 양자화(Q4_K_M, Q5_K_M, Q8_0) 비교 및 로컬 추론 벤치마크를 수행합니다.

| Item | Detail |
|------|--------|
| Task | LoRA 병합 → GGUF 변환 → 양자화 비교 |
| Base Model | Qwen/Qwen2.5-7B-Instruct |
| Adapter | LoRA (09에서 학습) |
| Quantization | F16, Q4_K_M, Q5_K_M, Q8_0 |
| Inference | llama-cpp-python |
| Environment | Local CPU (또는 Kaggle) |

---
## 1. Environment Setup

In [ ]:
%%capture
!pip install -q llama-cpp-python transformers peft accelerate plotly rouge-score bert-score sentencepiece kaleido

In [ ]:
import os, json, time, warnings, gc, subprocess, shutil
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'iframe'
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/civilcomplaint-processed'
    OUT_DIR = '/kaggle/working'
    IS_KAGGLE = True
else:
    DATA_DIR = '../data/processed'
    OUT_DIR = '..'
    IS_KAGGLE = False

RESULTS_DIR = os.path.join(OUT_DIR, 'results')
MODELS_DIR = os.path.join(OUT_DIR, 'models')
ADAPTER_DIR = os.path.join(MODELS_DIR, 'lora_adapter')
MERGED_DIR = os.path.join(MODELS_DIR, 'merged_model')
GGUF_DIR = os.path.join(MODELS_DIR, 'gguf')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

# Load LoRA results
lora_results = None
lora_path = os.path.join(RESULTS_DIR, 'generation_lora_results.json')
if os.path.exists(lora_path):
    with open(lora_path) as f:
        lora_results = json.load(f)
    print(f"LoRA results loaded: ROUGE-L={lora_results['metrics'].get('lora_rouge_l', 'N/A')}")

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Models: {MODELS_DIR}")
print(f"GGUF output: {GGUF_DIR}")

---
## 2. Merge LoRA Adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# Check if merged model already exists
if os.path.exists(os.path.join(MERGED_DIR, 'config.json')):
    print(f"Merged model already exists: {MERGED_DIR}")
    print("Skipping merge step.")
else:
    print(f"Loading base model: {MODEL_ID}...")
    t0 = time.time()
    
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="cpu",
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    
    print(f"Base model loaded: {time.time()-t0:.1f}s")
    
    # Load LoRA adapter
    if os.path.exists(os.path.join(ADAPTER_DIR, 'adapter_config.json')):
        print(f"Loading LoRA adapter: {ADAPTER_DIR}")
        model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
        
        # Merge and unload
        print("Merging LoRA weights into base model...")
        model = model.merge_and_unload()
        
        # Save merged model
        os.makedirs(MERGED_DIR, exist_ok=True)
        model.save_pretrained(MERGED_DIR)
        tokenizer.save_pretrained(MERGED_DIR)
        
        merge_time = time.time() - t0
        print(f"Merged model saved: {MERGED_DIR} ({merge_time:.1f}s)")
        
        # Cleanup
        del model, base_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print(f"LoRA adapter not found at {ADAPTER_DIR}")
        print("Saving base model as merged model for demo...")
        os.makedirs(MERGED_DIR, exist_ok=True)
        base_model.save_pretrained(MERGED_DIR)
        tokenizer.save_pretrained(MERGED_DIR)
        del base_model
        gc.collect()

# Check saved files
import glob as _glob
merged_files = _glob.glob(os.path.join(MERGED_DIR, '*'))
total_size = sum(os.path.getsize(f) for f in merged_files if os.path.isfile(f))
print(f"\nMerged model files: {len(merged_files)}")
print(f"Total size: {total_size/1e9:.1f} GB")

---
## 3. GGUF Conversion

In [ ]:
# --- Clone llama.cpp and convert to GGUF ---
LLAMA_CPP_DIR = os.path.join(OUT_DIR, 'llama.cpp')

if not os.path.exists(LLAMA_CPP_DIR):
    print("Cloning llama.cpp...")
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/ggerganov/llama.cpp', LLAMA_CPP_DIR],
        check=True, capture_output=True,
    )
    # Install conversion dependencies
    subprocess.run(
        ['pip', 'install', '-q', '-r', os.path.join(LLAMA_CPP_DIR, 'requirements.txt')],
        capture_output=True,
    )
    print("llama.cpp cloned and dependencies installed.")
else:
    print(f"llama.cpp already exists: {LLAMA_CPP_DIR}")

# Convert to F16 GGUF
f16_path = os.path.join(GGUF_DIR, 'model-f16.gguf')

if not os.path.exists(f16_path):
    print("\nConverting to GGUF (F16)...")
    convert_script = os.path.join(LLAMA_CPP_DIR, 'convert_hf_to_gguf.py')
    
    t0 = time.time()
    result = subprocess.run(
        ['python', convert_script, MERGED_DIR,
         '--outfile', f16_path,
         '--outtype', 'f16'],
        capture_output=True, text=True,
    )
    
    if result.returncode == 0:
        f16_size = os.path.getsize(f16_path) / 1e9
        print(f"F16 GGUF created: {f16_path} ({f16_size:.1f} GB, {time.time()-t0:.0f}s)")
    else:
        print(f"Conversion failed:\n{result.stderr[:500]}")
else:
    f16_size = os.path.getsize(f16_path) / 1e9
    print(f"F16 GGUF exists: {f16_path} ({f16_size:.1f} GB)")

In [ ]:
# --- Quantization ---
QUANT_TYPES = ['Q4_K_M', 'Q5_K_M', 'Q8_0']
quant_paths = {}
quant_sizes = {}

# Find llama-quantize binary
quantize_bin = os.path.join(LLAMA_CPP_DIR, 'build', 'bin', 'llama-quantize')
if not os.path.exists(quantize_bin):
    quantize_bin = shutil.which('llama-quantize')
    if not quantize_bin:
        # Try building
        print("Building llama.cpp quantize tool...")
        subprocess.run(
            ['cmake', '-B', 'build', '-DCMAKE_BUILD_TYPE=Release'],
            cwd=LLAMA_CPP_DIR, capture_output=True,
        )
        subprocess.run(
            ['cmake', '--build', 'build', '--config', 'Release', '-j', '--target', 'llama-quantize'],
            cwd=LLAMA_CPP_DIR, capture_output=True,
        )
        quantize_bin = os.path.join(LLAMA_CPP_DIR, 'build', 'bin', 'llama-quantize')

for qt in QUANT_TYPES:
    out_name = f'model-{qt.lower().replace("_", "-")}.gguf'
    out_path = os.path.join(GGUF_DIR, out_name)
    quant_paths[qt] = out_path
    
    if os.path.exists(out_path):
        size_gb = os.path.getsize(out_path) / 1e9
        quant_sizes[qt] = size_gb
        print(f"{qt}: {out_path} ({size_gb:.1f} GB) [exists]")
        continue
    
    if not os.path.exists(f16_path):
        print(f"F16 GGUF not found, skipping {qt}")
        continue
        
    if quantize_bin and os.path.exists(quantize_bin):
        print(f"\nQuantizing {qt}...")
        t0 = time.time()
        result = subprocess.run(
            [quantize_bin, f16_path, out_path, qt],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            size_gb = os.path.getsize(out_path) / 1e9
            quant_sizes[qt] = size_gb
            print(f"  {qt}: {size_gb:.1f} GB ({time.time()-t0:.0f}s)")
        else:
            print(f"  {qt} failed: {result.stderr[:200]}")
    else:
        print(f"llama-quantize not found, skipping {qt}")

# Add F16 to sizes
if os.path.exists(f16_path):
    quant_sizes['F16'] = os.path.getsize(f16_path) / 1e9

print(f"\n=== Model Sizes ===")
for name, size in sorted(quant_sizes.items()):
    print(f"  {name}: {size:.1f} GB")

In [ ]:
# --- Model size comparison chart ---
if quant_sizes:
    names = list(quant_sizes.keys())
    sizes = list(quant_sizes.values())
    
    # Sort by size descending
    sorted_pairs = sorted(zip(names, sizes), key=lambda x: x[1], reverse=True)
    names, sizes = zip(*sorted_pairs)
    
    colors = ['#ef5350' if 'F16' in n else '#42a5f5' if 'Q4' in n 
              else '#66bb6a' if 'Q5' in n else '#ffa726' for n in names]
    
    fig = go.Figure(go.Bar(
        x=list(names), y=list(sizes),
        marker_color=colors,
        text=[f'{s:.1f} GB' for s in sizes],
        textposition='outside',
    ))
    
    fig.update_layout(
        title='GGUF Model Size Comparison',
        yaxis_title='Size (GB)',
        width=700, height=450,
    )
    fig.show(renderer='iframe')
else:
    print("No quantized models available for comparison.")

---
## 4. Inference Benchmarking

In [ ]:
# --- Inference benchmark ---
from llama_cpp import Llama

# Load eval data
qa_path = os.path.join(DATA_DIR, 'qa_pairs.parquet')
qa_df = pd.read_parquet(qa_path)
N_BENCH = 50
np.random.seed(42)
bench_indices = np.random.choice(len(qa_df), size=N_BENCH, replace=False)
bench_df = qa_df.iloc[bench_indices].reset_index(drop=True)
bench_questions = bench_df['question'].tolist()
bench_gt_answers = bench_df['answer'].tolist()

PROMPT_FMT = "### 질문\n{question}\n\n### 답변\n"

bench_results = {}

for qt_name, qt_path in quant_paths.items():
    if not os.path.exists(qt_path):
        print(f"Skipping {qt_name}: file not found")
        continue
    
    print(f"\n=== Benchmarking {qt_name} ===")
    t0 = time.time()
    
    llm = Llama(
        model_path=qt_path,
        n_ctx=1024,
        n_threads=4,
        verbose=False,
    )
    load_time = time.time() - t0
    print(f"  Model loaded: {load_time:.1f}s")
    
    answers = []
    latencies = []
    total_tokens = 0
    
    for i, q in enumerate(bench_questions):
        prompt = PROMPT_FMT.format(question=q)
        
        t1 = time.time()
        output = llm(
            prompt,
            max_tokens=256,
            temperature=0.3,
            stop=["### 질문", "\n\n\n"],
            echo=False,
        )
        lat = time.time() - t1
        
        text = output['choices'][0]['text'].strip()
        n_tokens = output['usage']['completion_tokens']
        
        answers.append(text)
        latencies.append(lat)
        total_tokens += n_tokens
        
        if (i + 1) % 10 == 0:
            avg_lat = np.mean(latencies)
            tps = total_tokens / sum(latencies)
            print(f"  [{i+1}/{N_BENCH}] avg latency: {avg_lat:.2f}s, tokens/s: {tps:.1f}")
    
    total_time = sum(latencies)
    avg_tps = total_tokens / total_time if total_time > 0 else 0
    
    bench_results[qt_name] = {
        'answers': answers,
        'latencies': latencies,
        'avg_latency': float(np.mean(latencies)),
        'tokens_per_sec': float(avg_tps),
        'total_tokens': total_tokens,
        'load_time': load_time,
    }
    
    print(f"  Done: avg {np.mean(latencies):.2f}s/query, {avg_tps:.1f} tokens/s")
    
    # Cleanup
    del llm
    gc.collect()

print(f"\nBenchmark complete: {len(bench_results)} models tested")

---
## 5. Quality vs Speed

In [ ]:
# --- Compute quality metrics for each quantization ---
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

for qt_name, br in bench_results.items():
    answers = br['answers']
    
    # ROUGE-L
    rouge_scores = [scorer.score(gt, pred)['rougeL'].fmeasure 
                    for gt, pred in zip(bench_gt_answers, answers)]
    br['rouge_l'] = float(np.mean(rouge_scores))
    br['rouge_l_scores'] = rouge_scores
    
    # BERTScore
    try:
        _, _, F1 = bert_score_fn(answers, bench_gt_answers, lang='ko', verbose=False, batch_size=16)
        br['bertscore'] = float(F1.mean().item())
        br['bertscore_scores'] = F1.numpy().tolist()
    except Exception:
        br['bertscore'] = 0.0
        br['bertscore_scores'] = [0.0] * len(answers)
    
    print(f"{qt_name}: ROUGE-L={br['rouge_l']:.4f}, BERTScore={br['bertscore']:.4f}, "
          f"Speed={br['tokens_per_sec']:.1f} tok/s")

# --- Scatter: BERTScore vs Tokens/sec ---
if bench_results:
    qt_names = list(bench_results.keys())
    bert_vals = [bench_results[n]['bertscore'] for n in qt_names]
    speed_vals = [bench_results[n]['tokens_per_sec'] for n in qt_names]
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=speed_vals, y=bert_vals,
        mode='markers+text',
        text=qt_names,
        textposition='top center',
        marker=dict(size=15, color=['#42a5f5', '#66bb6a', '#ffa726'][:len(qt_names)]),
    ))
    
    fig.update_layout(
        title='Quality vs Speed: GGUF Quantization Comparison',
        xaxis_title='Tokens/sec',
        yaxis_title='BERTScore F1',
        width=700, height=450,
    )
    fig.show(renderer='iframe')

In [ ]:
# --- Grouped bar: quantization comparison ---
if bench_results:
    qt_names = list(bench_results.keys())
    
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=['ROUGE-L', 'BERTScore F1', 'Speed (tokens/s)'],
    )
    
    colors = ['#42a5f5', '#66bb6a', '#ffa726'][:len(qt_names)]
    
    fig.add_trace(go.Bar(
        x=qt_names,
        y=[bench_results[n]['rouge_l'] for n in qt_names],
        marker_color=colors,
        text=[f"{bench_results[n]['rouge_l']:.4f}" for n in qt_names],
        textposition='outside',
        showlegend=False,
    ), row=1, col=1)
    
    fig.add_trace(go.Bar(
        x=qt_names,
        y=[bench_results[n]['bertscore'] for n in qt_names],
        marker_color=colors,
        text=[f"{bench_results[n]['bertscore']:.4f}" for n in qt_names],
        textposition='outside',
        showlegend=False,
    ), row=1, col=2)
    
    fig.add_trace(go.Bar(
        x=qt_names,
        y=[bench_results[n]['tokens_per_sec'] for n in qt_names],
        marker_color=colors,
        text=[f"{bench_results[n]['tokens_per_sec']:.1f}" for n in qt_names],
        textposition='outside',
        showlegend=False,
    ), row=1, col=3)
    
    fig.update_layout(
        title='Quantization Comparison: Quality & Speed',
        width=1000, height=420,
        margin=dict(t=80),
    )
    fig.update_yaxes(range=[0, 1.1], row=1, col=1)
    fig.update_yaxes(range=[0, 1.1], row=1, col=2)
    fig.show(renderer='iframe')

---
## 6. Full Pipeline Comparison

In [ ]:
# --- Full pipeline progression chart ---
stages = []
scores = []
colors = []

# Load all previous results
bm25_path = os.path.join(RESULTS_DIR, 'retrieval_bm25_results.json')
sbert_path = os.path.join(RESULTS_DIR, 'retrieval_sbert_results.json')
rag_path = os.path.join(RESULTS_DIR, 'generation_rag_results.json')

if os.path.exists(bm25_path):
    with open(bm25_path) as f:
        bm25 = json.load(f)
    stages.append('BM25\n(Retrieval)')
    scores.append(bm25.get('recall_at_5', 0))
    colors.append('#ef5350')

if os.path.exists(sbert_path):
    with open(sbert_path) as f:
        sbert = json.load(f)
    stages.append('SBERT+FAISS\n(Retrieval)')
    scores.append(sbert.get('recall_at_5', 0))
    colors.append('#ff7043')

if os.path.exists(rag_path):
    with open(rag_path) as f:
        rag = json.load(f)
    stages.append('RAG\n(Generation)')
    scores.append(rag['metrics'].get('rag_bertscore', 0))
    colors.append('#ffa726')

if lora_results:
    stages.append('QLoRA\n(Fine-tuned)')
    scores.append(lora_results['metrics'].get('lora_bertscore', 0))
    colors.append('#66bb6a')

# Best GGUF
if bench_results:
    best_qt = max(bench_results.keys(), key=lambda n: bench_results[n]['bertscore'])
    stages.append(f'GGUF\n({best_qt})')
    scores.append(bench_results[best_qt]['bertscore'])
    colors.append('#42a5f5')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=stages, y=scores,
    marker_color=colors,
    text=[f'{v:.4f}' for v in scores],
    textposition='outside',
))

# Add connecting line
fig.add_trace(go.Scatter(
    x=stages, y=scores,
    mode='lines+markers',
    line=dict(color='gray', width=1, dash='dot'),
    marker=dict(size=6, color='gray'),
    showlegend=False,
))

fig.update_layout(
    title='Full Pipeline Progression: Retrieval → Generation → Deployment',
    yaxis_title='Score (Recall@5 / BERTScore F1)',
    yaxis_range=[0, max(scores) * 1.3 if scores else 1],
    width=900, height=500,
    annotations=[dict(
        text="Retrieval stages: Recall@5 | Generation stages: BERTScore F1",
        xref="paper", yref="paper", x=0.5, y=-0.12,
        showarrow=False, font=dict(size=11, color='gray'),
    )],
    margin=dict(b=80),
)
fig.show(renderer='iframe')

---
## 7. Save Results

In [ ]:
# --- Save generation_gguf_results.json ---
gguf_output = {
    'method': 'GGUF',
    'base_model': MODEL_ID,
    'quantizations': {},
    'benchmark': {
        'n_queries': N_BENCH,
        'prompt_format': PROMPT_FMT,
    },
}

for qt_name, br in bench_results.items():
    gguf_output['quantizations'][qt_name] = {
        'size_gb': round(quant_sizes.get(qt_name, 0), 2),
        'rouge_l': round(br['rouge_l'], 4),
        'bertscore': round(br['bertscore'], 4),
        'tokens_per_sec': round(br['tokens_per_sec'], 1),
        'avg_latency_s': round(br['avg_latency'], 2),
        'load_time_s': round(br['load_time'], 1),
    }

if bench_results:
    best_qt = max(bench_results.keys(), key=lambda n: bench_results[n]['bertscore'])
    gguf_output['best_quantization'] = best_qt
    gguf_output['best_bertscore'] = round(bench_results[best_qt]['bertscore'], 4)
    gguf_output['best_tokens_per_sec'] = round(bench_results[best_qt]['tokens_per_sec'], 1)

# Pipeline context
gguf_output['pipeline_scores'] = {
    'lora_bertscore': lora_results['metrics'].get('lora_bertscore') if lora_results else None,
}

results_path = os.path.join(RESULTS_DIR, 'generation_gguf_results.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(gguf_output, f, ensure_ascii=False, indent=2)

print(f"Results saved: {results_path}")
print(json.dumps(gguf_output, ensure_ascii=False, indent=2))

In [ ]:
# --- Base64 download ---
import base64, zipfile, io

def create_download_link(filepath, filename=None):
    if filename is None:
        filename = filepath.split('/')[-1]
    with open(filepath, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / 1024 / 1024
    href = (f'<a href="data:application/octet-stream;base64,{b64}" '
            f'download="{filename}">'
            f'Download: {filename} ({size_mb:.1f} MB)</a>')
    display(HTML(href))


# Results JSON
create_download_link(results_path)

# GGUF files (note: these may be too large for Base64 in browser)
for qt_name, qt_path in quant_paths.items():
    if os.path.exists(qt_path):
        size_gb = os.path.getsize(qt_path) / 1e9
        if size_gb < 2.0:  # Only download links for files < 2GB
            create_download_link(qt_path)
        else:
            print(f"{qt_name}: {size_gb:.1f} GB (too large for browser download)")

---
## 8. Summary

In [ ]:
print("=" * 70)
print("     10. GGUF Quantization & Deployment -- Summary")
print("=" * 70)
print()
print(f"  Base Model: {MODEL_ID}")
print(f"  Merged:     {MERGED_DIR}")
print(f"  GGUF:       {GGUF_DIR}")
print()

# Quantization comparison table
if bench_results:
    print(f"  {'Quant':<10} {'Size':>8} {'ROUGE-L':>10} {'BERTScore':>10} {'Speed':>12} {'Latency':>10}")
    print(f"  {'-'*60}")
    for qt_name in sorted(bench_results.keys()):
        br = bench_results[qt_name]
        size = quant_sizes.get(qt_name, 0)
        print(f"  {qt_name:<10} {size:>7.1f}G {br['rouge_l']:>10.4f} {br['bertscore']:>10.4f} "
              f"{br['tokens_per_sec']:>10.1f}t/s {br['avg_latency']:>9.2f}s")
    
    best_qt = max(bench_results.keys(), key=lambda n: bench_results[n]['bertscore'])
    print(f"\n  Best quality:  {best_qt} (BERTScore={bench_results[best_qt]['bertscore']:.4f})")
    
    fastest = max(bench_results.keys(), key=lambda n: bench_results[n]['tokens_per_sec'])
    print(f"  Fastest:       {fastest} ({bench_results[fastest]['tokens_per_sec']:.1f} tokens/s)")

print()
print(f"  Artifacts:")
print(f"    Merged model -> {MERGED_DIR}")
print(f"    GGUF files   -> {GGUF_DIR}")
print(f"    Results      -> {results_path}")
print()
print("  ===== Pipeline Complete =====")
print("  01 Data Exploration")
print("  02 Preprocessing")
print("  03 Task Definition")
print("  04 Baseline ML Classification")
print("  05 Deep Classification")
print("  06 BM25 Retrieval")
print("  07 SBERT + FAISS")
print("  08 RAG Pipeline")
print("  09 QLoRA Fine-tuning")
print("  10 GGUF Quantization & Deployment  ← YOU ARE HERE")
print("=" * 70)

---
## 9. 양자화 수준 선택 근거

### GGUF 양자화 방식 비교

| Quantization | Bits | Method | Characteristics |
|-------------|------|--------|----------------|
| **F16** | 16 | Full precision | 기준선. 최고 품질, 최대 크기 (~14GB for 7B) |
| **Q8_0** | 8 | Round-to-nearest | 품질 손실 거의 없음 (~0.1% degradation). 크기 50% 절감 |
| **Q5_K_M** | 5 | K-quant mixed | 중요 레이어(attention)는 6bit, 나머지 5bit. 품질-크기 균형점 |
| **Q4_K_M** | 4 | K-quant mixed | 가장 공격적 양자화. 크기 ~70% 절감. 약간의 품질 손실 허용 |

### K-Quant의 핵심 원리

`K_M` (K-quant Medium)은 **레이어 중요도에 따라 bit 할당을 차등화**합니다:
- Attention의 Q/K/V projection → 높은 bit (6bit)
- FFN의 gate/up/down → 낮은 bit (4~5bit)
- 이는 uniform quantization 대비 동일 크기에서 더 높은 품질을 보장합니다.

### 배포 시나리오별 추천

| Scenario | Recommended | Reason |
|----------|------------|--------|
| **프로덕션 서버 (GPU)** | Q5_K_M | 품질과 속도의 최적 균형. 8GB VRAM에서 운영 가능 |
| **경량 서버 (CPU only)** | Q4_K_M | 최소 메모리로 운영. CPU에서도 합리적 속도 |
| **품질 최우선** | Q8_0 | 품질 손실 최소화가 필수인 경우 (의료/법률 등) |
| **개발/테스트** | Q4_K_M | 빠른 반복 테스트용 |

In [ ]:
# === 전체 파이프라인 개선율 요약 (Classification → Retrieval → Generation → Deployment) ===
print("=" * 75)
print("  전체 파이프라인 성능 개선 요약 (End-to-End)")
print("=" * 75)

# --- Classification ---
ml_path = os.path.join(RESULTS_DIR, 'classification_ml_results.json')
dl_path = os.path.join(RESULTS_DIR, 'classification_dl_results.json')
bm25_path_r = os.path.join(RESULTS_DIR, 'retrieval_bm25_results.json')
sbert_path_r = os.path.join(RESULTS_DIR, 'retrieval_sbert_results.json')
rag_path_r = os.path.join(RESULTS_DIR, 'generation_rag_results.json')
lora_path_r = os.path.join(RESULTS_DIR, 'generation_lora_results.json')
gguf_path_r = os.path.join(RESULTS_DIR, 'generation_gguf_results.json')

print("\n  [Phase 1] Classification (Domain)")
if os.path.exists(ml_path) and os.path.exists(dl_path):
    with open(ml_path) as f:
        ml_r = json.load(f)
    with open(dl_path) as f:
        dl_r = json.load(f)
    ml_f1 = ml_r['best_models']['domain']['macro_f1']
    dl_f1 = dl_r['best_models']['domain']['macro_f1']
    print(f"    ML Baseline (TF-IDF):  Macro F1 = {ml_f1:.4f}")
    print(f"    DL (KoELECTRA):        Macro F1 = {dl_f1:.4f}  ({dl_f1 - ml_f1:+.4f}, {(dl_f1-ml_f1)/ml_f1*100:+.1f}%)")
else:
    print("    Results not available")

print("\n  [Phase 2] Retrieval")
if os.path.exists(bm25_path_r) and os.path.exists(sbert_path_r):
    with open(bm25_path_r) as f:
        bm25_r = json.load(f)
    with open(sbert_path_r) as f:
        sbert_r = json.load(f)
    bm25_recall = bm25_r.get('recall_at_5', bm25_r.get('metrics', {}).get('recall_at_5', 0))
    sbert_recall = sbert_r.get('recall_at_5', 0)
    print(f"    BM25:                  Recall@5 = {bm25_recall:.4f}")
    print(f"    SBERT+FAISS:           Recall@5 = {sbert_recall:.4f}  ({sbert_recall - bm25_recall:+.4f}, {(sbert_recall-bm25_recall)/bm25_recall*100 if bm25_recall > 0 else 0:+.1f}%)")
else:
    print("    Results not available")

print("\n  [Phase 3] Generation")
if os.path.exists(rag_path_r):
    with open(rag_path_r) as f:
        rag_r = json.load(f)
    zs_bs = rag_r['metrics'].get('zs_bertscore', 0)
    rag_bs = rag_r['metrics'].get('rag_bertscore', 0)
    print(f"    Zero-shot:             BERTScore = {zs_bs:.4f}")
    print(f"    RAG:                   BERTScore = {rag_bs:.4f}  ({rag_bs - zs_bs:+.4f})")

if os.path.exists(lora_path_r):
    with open(lora_path_r) as f:
        lora_r = json.load(f)
    base_bs_val = lora_r['metrics'].get('base_bertscore', 0)
    lora_bs_val = lora_r['metrics'].get('lora_bertscore', 0)
    print(f"    Base (Qwen2.5-7B):     BERTScore = {base_bs_val:.4f}")
    print(f"    QLoRA Fine-tuned:      BERTScore = {lora_bs_val:.4f}  ({lora_bs_val - base_bs_val:+.4f})")

print("\n  [Phase 4] Deployment (GGUF Quantization)")
if bench_results:
    for qt_name in sorted(bench_results.keys()):
        br = bench_results[qt_name]
        size = quant_sizes.get(qt_name, 0)
        print(f"    {qt_name:<10}  BERTScore={br['bertscore']:.4f}  Size={size:.1f}GB  Speed={br['tokens_per_sec']:.1f}t/s")

print("\n" + "=" * 75)
print("  Pipeline Complete: Data → Classification → Retrieval → Generation → Deploy")
print("=" * 75)

In [ ]:
# === Kaggle Dataset 자동 업로드 ===
if IS_KAGGLE:
    UPLOAD_DIR = '/kaggle/working/dataset_upload'
    os.makedirs(UPLOAD_DIR, exist_ok=True)

    # 결과 파일 심볼릭 링크
    for src in [results_path]:
        dst = os.path.join(UPLOAD_DIR, os.path.basename(src))
        if os.path.exists(dst):
            os.remove(dst)
        os.symlink(src, dst)

    # GGUF 파일 심볼릭 링크 (가장 작은 Q4_K_M만 업로드)
    q4_path = quant_paths.get('Q4_K_M')
    if q4_path and os.path.exists(q4_path):
        dst = os.path.join(UPLOAD_DIR, os.path.basename(q4_path))
        if os.path.exists(dst):
            os.remove(dst)
        os.symlink(q4_path, dst)

    meta = {
        "title": "civilcomplaint-gguf-deploy",
        "id": "kukass/civilcomplaint-gguf-deploy",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(os.path.join(UPLOAD_DIR, 'dataset-metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    !kaggle datasets create -p {UPLOAD_DIR} --dir-mode zip
    print("✅ Kaggle 데이터셋 업로드 완료: civilcomplaint-gguf-deploy")
else:
    print("ℹ️ 로컬 환경 — Kaggle 업로드 건너뜀")